<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%2010/Question_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Q1. Prior Belief Boundaries

The remaining stiffness efficiency factor ($\Theta$) is physically restricted to the interval:

$$0 < \theta \leq 1$$

Therefore, a Beta distribution is an appropriate choice because it is naturally bounded between 0 and 1.

The initial prior distribution is:

$$\Theta \sim Beta(8,1.5)$$

The probability density function is:

$$f_{\Theta}^{(0)}(\theta)=\frac{1}{B(8,1.5)}\theta^{8-1}(1-\theta)^{1.5-1}$$

where:

* ($\alpha=8$)
* ($\beta=1.5$)

The expected value of a Beta distribution is:

$$E[\Theta]=\frac{\alpha}{\alpha+\beta}$$

Therefore:

$$E[\Theta^{(0)}] = \frac{8}{8+1.5}$$

$$\boxed{E[\Theta^{(0)}]=0.8421}$$

This means that before collecting any sensor measurements, engineers expect the component to retain approximately **84.21% of its original stiffness**.

The Beta(8,1.5) distribution is suitable because:

* The distribution is concentrated close to ($\theta=1$).
* It represents an initially healthy component.
* It assigns lower probability to severe degradation states.
* It incorporates previous manufacturing and historical inspection information.

However, the prior is not completely fixed at one, allowing the Bayesian system to update the belief if sensor measurements indicate degradation.

---

# Q2. Structural Likelihood Formulation

The sensor measurement model is:

$$y_k=\theta K_{\text{nominal}}e^{\epsilon_k}$$

where:

$$\epsilon_k \sim N(0,\sigma^2)$$

Taking logarithms:

$$\ln(y_k)=\ln(\theta K_{\text{nominal}})+\epsilon_k$$

Therefore:

$$\ln(y_k) \sim N(\ln(\theta K_{\text{nominal}}),\sigma^2)$$

Using the log-normal probability density function, the likelihood contribution of a single sensor measurement is:

$$L(y_k|\theta) = \frac{1}{y_k\sigma\sqrt{2\pi}}\exp\left[-\frac{(\ln(y_k)-\ln(\theta K_{\text{nominal}}))^2}{2\sigma^2}\right]$$

where:

* ($y_k$) is the measured stiffness.
* ($K_{\text{nominal}}$) is the healthy stiffness.
* ($\theta$) is the unknown stiffness efficiency factor.
* ($\sigma$) represents measurement uncertainty.

---

For a sequence of sensor measurements:

$$\mathbf y^{(k)} = (y_1,y_2,\dots,y_k)$$

the joint likelihood is:

$$L(\mathbf y^{(k)}|\theta) = \prod_{i=1}^{k}L(y_i|\theta)$$

Therefore:

$$\boxed{L(\mathbf y^{(k)}|\theta) = \prod_{i=1}^{k}\frac{1}{y_i\sigma\sqrt{2\pi}}\exp\left[-\frac{(\ln(y_i)-\ln(\theta K_{\text{nominal}}))^2}{2\sigma^2}\right]}$$

---

# Q3. Mathematical Formulation of the Non-Conjugate Grid Update

The initial prior is:

$$\Theta\sim Beta(8,1.5)$$

The likelihood function is log-normal.

Unlike the Beta-Binomial model, the Beta distribution is **not conjugate** to the log-normal likelihood.

Therefore:

* Multiplying the Beta prior with the likelihood does not produce another Beta distribution.
* No closed-form posterior parameter update exists.
* Numerical approximation methods are required.

The Bayesian update equation is:

$$f_{\Theta|\mathbf Y^{(k)}}(\theta|\mathbf y^{(k)}) \propto L(y_k|\theta)f_{\Theta|\mathbf Y^{(k-1)}}(\theta|\mathbf y^{(k-1)})$$

Substituting the likelihood:

$$f_{\Theta|\mathbf Y^{(k)}}(\theta|\mathbf y^{(k)}) \propto \frac{1}{y_k\sigma\sqrt{2\pi}}\exp\left[-\frac{(\ln(y_k)-\ln(\theta K_{\text{nominal}}))^2}{2\sigma^2}\right]f_{\Theta|\mathbf Y^{(k-1)}}(\theta)$$

After multiplication, the posterior must be normalized numerically.

---

# Q4. Running Point Estimates

Since the posterior does not have a closed-form solution, the estimates are obtained using numerical integration over:

$$0<\theta\leq1$$

---

## Posterior Mean (Bayesian Estimate)

The posterior mean is:

$$\boxed{\widehat{\theta}_{Bayes}^{(k)} = \int_0^1 \theta f_{\Theta|\mathbf Y^{(k)}}(\theta|\mathbf y^{(k)}) d\theta}$$

This represents the expected remaining stiffness factor after observing the sensor data.

---

## MAP Estimate

The Maximum A Posteriori estimate is:

$$\boxed{\widehat{\theta}_{MAP}^{(k)} = \arg\max_{\theta\in(0,1]} f_{\Theta|\mathbf Y^{(k)}}(\theta|\mathbf y^{(k)})}$$

It corresponds to the stiffness value having the highest posterior probability.

---

# Q5. Algorithmic Grid Approximation and Normalization

A numerical grid method is used to approximate the posterior distribution.

The procedure is:

### Step 1: Create a bounded stiffness grid

Because the physical range is:

$$0<\theta\leq1$$

a grid is created:

$$\theta_1,\theta_2,...,\theta_m$$

For example:

$$\theta\in[0.01,1]$$

The lower boundary is not chosen as zero because:

* ($\ln(0)$) is undefined.
* Zero stiffness represents complete failure.

---

### Step 2: Initialize Prior

Evaluate the Beta prior on every grid point:

$$p_0(\theta_i)=Beta(\theta_i;8,1.5)$$

Normalize:

$$\int_0^1p_0(\theta)d\theta=1$$

using numerical integration.

---

### Step 3: Receive New Sensor Measurement

For a new measurement ($y_k$):

Calculate the likelihood at every grid point:

$$L(y_k|\theta_i)$$

---

### Step 4: Update Posterior

Multiply the previous posterior by the likelihood:

$$p_k(\theta_i) = p_{k-1}(\theta_i)L(y_k|\theta_i)$$

This gives the unnormalized posterior.

---

### Step 5: Normalize Using Trapezoidal Rule

The normalization constant is:

$$Z = \int_0^1p_k(\theta)d\theta$$

Numerically:

$$Z\approx np.trapezoid(p_k,\theta)$$

The normalized posterior becomes:

$$\boxed{p_k(\theta_i)=\frac{p_k(\theta_i)}{Z}}$$

This process is repeated for every new sensor measurement.

---

# Q6. Performance Tracking and Degradation Convergence Analysis

The true remaining stiffness is:

$$\theta_{true}=0.68$$

The nominal stiffness is:

$$K_{nominal}=50,\text{kN/mm}$$

The sensor model is:

$$y_k=0.68(50)e^{\epsilon_k}$$

where:

$$\epsilon_k\sim N(0,0.15^2)$$

The simulation process is:

1. Generate 15 sensor measurements.
2. Start with the optimistic prior (Beta(8,1.5)).
3. Update the posterior after every measurement.
4. Calculate:

   * Posterior Mean.
   * MAP estimate.
5. Plot posterior evolution and estimator convergence.

---

## Analysis

Initially, the posterior distribution is strongly concentrated near ($\theta=1$) because of the healthy prior assumption.

After receiving sensor measurements:

* Measurements lower than the nominal stiffness provide evidence of degradation.
* The posterior distribution gradually shifts toward smaller stiffness values.
* The posterior mean and MAP estimates decrease toward the true value:

$$\theta_{true}=0.68$$

After several measurements:

* The influence of the initial prior becomes weaker.
* The likelihood contribution from the sensor data dominates.
* The posterior distribution becomes concentrated around the actual degradation state.

Typically, around **5–10 sensor measurements** are sufficient for the posterior to move away from the optimistic healthy assumption and approach the 68% stiffness state.

The narrowing of the posterior density curves indicates reduced uncertainty in the estimated stiffness factor.

For structural health monitoring, this has an important engineering interpretation:

* A narrow posterior means engineers have higher confidence in the estimated damage level.
* If the posterior moves below a safety threshold, maintenance actions can be triggered.
* Continuous Bayesian updating enables early detection of structural degradation before catastrophic failure.

Thus, the bounded grid Bayesian method provides a practical framework for real-time structural condition assessment when analytical posterior solutions are unavailable.

In [1]:
# ================================================================
# Bayesian Structural Health Monitoring Using Grid Approximation
# Non-Conjugate Posterior Updating with Log-Normal Sensor Noise
# ================================================================

import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go


# ------------------------------------------------
# System Parameters
# ------------------------------------------------

np.random.seed(10)

# True hidden structural condition
theta_actual = 0.68

# Structural properties
K0 = 50.0          # Healthy stiffness (kN/mm)
sensor_sigma = 0.15

# Number of inspections
N = 15


# ------------------------------------------------
# Define bounded theta grid
# theta cannot be exactly zero because log(theta)
# is undefined in the likelihood calculation
# ------------------------------------------------

theta_values = np.linspace(0.01, 1.0, 800)


# ------------------------------------------------
# Initial Prior Distribution
# Healthy structure assumption:
# Theta ~ Beta(8,1.5)
# ------------------------------------------------

alpha_prior = 8
beta_prior = 1.5

posterior_density = stats.beta.pdf(
    theta_values,
    alpha_prior,
    beta_prior
)

# Normalize the prior
posterior_density = (
    posterior_density /
    np.trapezoid(posterior_density, theta_values)
)


# ------------------------------------------------
# Store posterior distributions at selected times
# ------------------------------------------------

saved_posteriors = {
    0: posterior_density.copy()
}

inspection_points = [1, 2, 5, 10, 15]


# Store estimator history
bayes_estimates = [np.trapezoid(
    theta_values * posterior_density,
    theta_values
)]

map_estimates = [
    theta_values[np.argmax(posterior_density)]
]


# ------------------------------------------------
# Sequential Bayesian Updating
# ------------------------------------------------

sensor_data = []


for step in range(1, N + 1):

    # Generate sensor measurement
    random_error = np.random.normal(
        0,
        sensor_sigma
    )

    measured_stiffness = (
        theta_actual *
        K0 *
        np.exp(random_error)
    )

    sensor_data.append(measured_stiffness)


    # ------------------------------------------------
    # Compute likelihood over theta grid
    # y ~ LogNormal(log(theta*K0), sigma)
    # ------------------------------------------------

    expected_stiffness = theta_values * K0

    likelihood = stats.lognorm.pdf(
        measured_stiffness,
        s=sensor_sigma,
        scale=expected_stiffness
    )


    # ------------------------------------------------
    # Bayesian update:
    # New posterior ∝ old posterior × likelihood
    # ------------------------------------------------

    posterior_density *= likelihood


    # Numerical normalization
    area = np.trapezoid(
        posterior_density,
        theta_values
    )

    posterior_density /= area


    # ------------------------------------------------
    # Calculate Bayesian estimators
    # ------------------------------------------------

    posterior_mean = np.trapezoid(
        theta_values * posterior_density,
        theta_values
    )

    posterior_map = theta_values[
        np.argmax(posterior_density)
    ]


    bayes_estimates.append(
        posterior_mean
    )

    map_estimates.append(
        posterior_map
    )


    # Save density evolution
    if step in inspection_points:
        saved_posteriors[step] = posterior_density.copy()



# ================================================================
# Plot 1: Posterior Density Evolution
# ================================================================

density_plot = go.Figure()


for step, density in saved_posteriors.items():

    label = (
        "Initial Prior"
        if step == 0
        else f"Inspection {step}"
    )

    density_plot.add_trace(
        go.Scatter(
            x=theta_values,
            y=density,
            mode="lines",
            name=label
        )
    )


density_plot.add_vline(
    x=theta_actual,
    line_dash="dash",
    line_width=3,
    annotation_text="True θ = 0.68"
)


density_plot.update_layout(

    title="Evolution of Structural Condition Posterior",

    xaxis_title=
    "Remaining Stiffness Efficiency Factor θ",

    yaxis_title=
    "Posterior Probability Density",

    template="plotly_white"
)


density_plot.show()



# ================================================================
# Plot 2: Estimator Convergence
# ================================================================

steps = np.arange(0, N + 1)

estimate_plot = go.Figure()


estimate_plot.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_estimates,
        mode="lines+markers",
        name="Bayesian Posterior Mean"
    )
)


estimate_plot.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)


estimate_plot.add_trace(
    go.Scatter(
        x=steps,
        y=[theta_actual]*(N+1),
        mode="lines",
        name="True Structural State (0.68)"
    )
)


estimate_plot.update_layout(

    title=
    "Convergence of Structural Damage Estimates",

    xaxis_title=
    "Inspection Number",

    yaxis_title=
    "Estimated Remaining Stiffness θ",

    template="plotly_white"
)


estimate_plot.show()



# ------------------------------------------------
# Display Generated Sensor Measurements
# ------------------------------------------------

print("Generated sensor measurements (kN/mm):")

for i, value in enumerate(sensor_data, start=1):
    print(
        f"Inspection {i}: {value:.3f} kN/mm"
    )

Generated sensor measurements (kN/mm):
Inspection 1: 41.517 kN/mm
Inspection 2: 37.851 kN/mm
Inspection 3: 26.965 kN/mm
Inspection 4: 33.957 kN/mm
Inspection 5: 37.321 kN/mm
Inspection 6: 30.519 kN/mm
Inspection 7: 35.381 kN/mm
Inspection 8: 34.558 kN/mm
Inspection 9: 34.022 kN/mm
Inspection 10: 33.121 kN/mm
Inspection 11: 36.282 kN/mm
Inspection 12: 40.724 kN/mm
Inspection 13: 29.418 kN/mm
Inspection 14: 39.670 kN/mm
Inspection 15: 35.186 kN/mm
